In [8]:
import requests
from lakehouse.daft import bronze
import daft

In [9]:
CATALOG = "daft_catalog"

# 1. Set Up

In [10]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze",
}

# 2 Overwrite

In [11]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return daft.from_pylist(results)

    def target_path(self, table: str) -> str:
        return f"D:/Data/{self.catalog}/{self.target_schema}/{table}"


bronze_instance = StarWarsBronze(**options)

In [12]:
(
    bronze_instance.load()
    .transform()
    .write(mode="overwrite")
    .execute("people", "planets")
)

2025-03-30 21:59:59 | people | execute | Started
2025-03-30 21:59:59 | people | load | Started
2025-03-30 22:00:05 | people | load | Completed in 0.08 min
2025-03-30 22:00:05 | people | transform | Started
2025-03-30 22:00:05 | people | transform | Completed in 0.0 min
2025-03-30 22:00:05 | people | write | Started
c:\Users\nikol\miniconda3\envs\lh-exec\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-03-30 22:00:06 | people | write | Completed in 0.0 min
2025-03-30 22:00:06 | people | execute | Completed in 0.1 min
2025-03-30 22:00:06 | planets | execute | Started
2025-03-30 22:00:06 | planets | load | Started
2025-03-30 22:00:10 | planets | load | Completed in 0.05 min
2025-03-30 22:00:10 | planets | transform | Started
2025-03-30 22:00:10 | planets | transform | Completed in 0.0 min
2025-03-30 22:00:10 | pl

In [13]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/people")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8
2025-03-30 22:00:05.839054,Luke Skywalker,1,https://www.swapi.tech/api/people/1
2025-03-30 22:00:05.839054,C-3PO,2,https://www.swapi.tech/api/people/2
2025-03-30 22:00:05.839054,R2-D2,3,https://www.swapi.tech/api/people/3
2025-03-30 22:00:05.839054,Darth Vader,4,https://www.swapi.tech/api/people/4
2025-03-30 22:00:05.839054,Leia Organa,5,https://www.swapi.tech/api/people/5
2025-03-30 22:00:05.839054,Owen Lars,6,https://www.swapi.tech/api/people/6
2025-03-30 22:00:05.839054,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7
2025-03-30 22:00:05.839054,R5-D4,8,https://www.swapi.tech/api/people/8


No. Rows: 82


In [14]:
df = daft.read_deltalake(f"D:/Data/{CATALOG}/bronze/planets")
df.show()
print(f"No. Rows: {df.count_rows()}")

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8
2025-03-30 22:00:10.004817,Tatooine,1,https://www.swapi.tech/api/planets/1
2025-03-30 22:00:10.004817,Alderaan,2,https://www.swapi.tech/api/planets/2
2025-03-30 22:00:10.004817,Yavin IV,3,https://www.swapi.tech/api/planets/3
2025-03-30 22:00:10.004817,Hoth,4,https://www.swapi.tech/api/planets/4
2025-03-30 22:00:10.004817,Dagobah,5,https://www.swapi.tech/api/planets/5
2025-03-30 22:00:10.004817,Bespin,6,https://www.swapi.tech/api/planets/6
2025-03-30 22:00:10.004817,Endor,7,https://www.swapi.tech/api/planets/7
2025-03-30 22:00:10.004817,Naboo,8,https://www.swapi.tech/api/planets/8


No. Rows: 60


In [15]:
bronze_instance.data["people"].show()

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8
2025-03-30 22:00:05.839054,Luke Skywalker,1,https://www.swapi.tech/api/people/1
2025-03-30 22:00:05.839054,C-3PO,2,https://www.swapi.tech/api/people/2
2025-03-30 22:00:05.839054,R2-D2,3,https://www.swapi.tech/api/people/3
2025-03-30 22:00:05.839054,Darth Vader,4,https://www.swapi.tech/api/people/4
2025-03-30 22:00:05.839054,Leia Organa,5,https://www.swapi.tech/api/people/5
2025-03-30 22:00:05.839054,Owen Lars,6,https://www.swapi.tech/api/people/6
2025-03-30 22:00:05.839054,Beru Whitesun lars,7,https://www.swapi.tech/api/people/7
2025-03-30 22:00:05.839054,R5-D4,8,https://www.swapi.tech/api/people/8


In [16]:
bronze_instance.data["planets"].show()

"LH_BronzeTSTimestamp(Microseconds, None)",nameUtf8,uidUtf8,urlUtf8
2025-03-30 22:00:10.004817,Tatooine,1,https://www.swapi.tech/api/planets/1
2025-03-30 22:00:10.004817,Alderaan,2,https://www.swapi.tech/api/planets/2
2025-03-30 22:00:10.004817,Yavin IV,3,https://www.swapi.tech/api/planets/3
2025-03-30 22:00:10.004817,Hoth,4,https://www.swapi.tech/api/planets/4
2025-03-30 22:00:10.004817,Dagobah,5,https://www.swapi.tech/api/planets/5
2025-03-30 22:00:10.004817,Bespin,6,https://www.swapi.tech/api/planets/6
2025-03-30 22:00:10.004817,Endor,7,https://www.swapi.tech/api/planets/7
2025-03-30 22:00:10.004817,Naboo,8,https://www.swapi.tech/api/planets/8


# 6 Clean Up

In [17]:
import shutil

shutil.rmtree(f"D:/Data/{CATALOG}")